In [7]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import plotly.express as px

# Task 1 · Part 1 — From headlines to a dataset

**Guiding question:** Can Peruvian news headlines be transformed into a reliable news classifier?

The performance of any supervised classifier is ultimately bounded by the quality of the data on which it is trained. Accordingly, this first notebook addresses the data-generation stage of the project: it collects real headlines from a Peruvian newspaper, cleans and normalizes them, and assigns each item to one of four predefined thematic categories. In doing so, it establishes the empirical foundation upon which the subsequent modelling stage rests.

| Parameter | Value |
|---|---|
| **Download date** | 2026-10-18 |
| **Outlet** | Option B — El Comercio (`elcomercio.pe`) |
| **Source of article URLs** | Official section sitemaps: `https://elcomercio.pe/sitemap/web/{section}/?outputType=xml` |
| **Source of article fields** | Metadata of each article page (`og:title`, `og:description`, `article:published_time`) |
| **Sections retrieved** | `politica`, `economia`, `deporte-total`, `tecnologia` |

**Methodological note on the choice of outlet.** La República (Option A) was initially considered. However, a preliminary assessment (see below) revealed that each of its RSS feeds exposes only around 21 items, a volume insufficient to reach the required 90–100 news items per category. Consequently, El Comercio was selected, since its section sitemaps list approximately 100 articles each.

**Ethical considerations.** El Comercio's `robots.txt` explicitly lists these sitemaps for automated access and does not restrict article pages. Furthermore, a pause is introduced between consecutive requests in order to minimize the load imposed on the outlet's servers, in line with responsible web-scraping practices.

In [8]:
import time
from datetime import date

import pandas as pd
import requests
from bs4 import BeautifulSoup
import plotly.express as px

In [9]:
# =============================================
#  CONFIGURATION
# =============================================

fecha_descarga = date.today().isoformat()

# A descriptive User-Agent tells the outlet who is making the requests and why.
headers = {"User-Agent": "Mozilla/5.0 (Diplomado PUCP - Python Task 1, academic use)"}

# The assignment requires pauses of 1 to 2 seconds between requests.
pausa_segundos = 1.5

# Final source: El Comercio section sitemaps.
secciones = ["politica", "economia", "deporte-total", "tecnologia"]
url_sitemap = "https://elcomercio.pe/sitemap/web/{seccion}/?outputType=xml"

# Kept only for the preliminary assessment of La República (Option A).
url_rss = "https://larepublica.pe/rss/{seccion}.xml"

fecha_descarga

'2026-10-18'

All parameters are centralized in a single configuration cell, which enhances the reproducibility of the notebook: should the collection need to be repeated, only this cell must be modified. In this regard, the descriptive `User-Agent` transparently identifies the academic purpose of the requests, and the 1.5-second pause complies with the assignment's guideline of waiting between one and two seconds per request. Moreover, the `{seccion}` placeholder allows the same URL template to be reused for each of the four sections. Finally, the La República RSS template is retained solely to document the preliminary assessment that motivated the change of outlet.

## 1. Preliminary assessment: La República (Option A)

Before committing to a data source, its capacity to supply the required volume of news was assessed empirically.

In [10]:
# Test a single section before looping over all four.
respuesta = requests.get(url_rss.format(seccion="politica"), headers=headers, timeout=30)
respuesta.status_code

200

A single test request is sent to La República's politics RSS feed. The resulting status code indicates whether access was granted: `200` confirms a successful request, whereas `403` would signal that the outlet blocked it and `429` that requests were being sent too frequently.

In [11]:
# RSS is XML: each news article lives inside an <item> tag.
sopa = BeautifulSoup(respuesta.content, "xml")
items = sopa.find_all("item")
len(items)

21

Since RSS feeds are structured as XML documents, each news article is enclosed in an `<item>` tag, and `find_all("item")` therefore returns every article available in the feed. Although access was granted (status `200`), the feed exposes only 21 items. Extrapolating to the four sections, the RSS channel would yield roughly 84 news items, well below the required 350. The outlet's sitemaps and section pages were also inspected manually, yet none of them exposed more than about 16 items per section. **Accordingly, El Comercio (Option B) was adopted as the data source**, as its section sitemaps list approximately 100 articles each.